# Model Selection& Routing

**Module 13 · Lesson 13.3**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Why One Model For Everything Is Wrong
- Select The Model: Measure, Do Not Guess
- Rule-Based Routing
- Semantic / Classifier Routing
- Cascade Routing
- The Routing Gateway + Fallback

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install anthropic -q

# Reproducibility - the lesson uses seeded random values throughout

## One frontier model for every query, and the easy ones pay for it

> 💡 **Info**
>
> Your support assistant works, so nobody looks at how it routes. But it sends **every** request to the same frontier model — the most expensive one you have — whether the query is “what are your hours?” or “walk me through why this distributed lock is safe under a network partition.” The first needs a small model that costs a fraction of a cent; the second needs the frontier. Sending both to the frontier means the easy query, which is the *vast majority* of your traffic, is marked up **twenty to a hundred times** for a quality difference you cannot even measure on it. Lesson 13.1 gave you the law — cost is tokens times price — and 13.2 cut the tokens. This lesson cuts the *price*, by matching each query to the cheapest model that can still answer it well. You will map the price-performance frontier, **select** a model per task by measuring an eval set rather than guessing, **route** the obvious cases for free with rules and the ambiguous ones with a classifier, **escalate** only the hard ones with a cheap-first cascade, put it all behind a **gateway with fallbacks**, and — the step everyone skips — **measure the router itself** so a silently drifting route cannot quietly wreck quality while the bill still looks great.

### 🎯 What you will be able to do after this lesson

- **Build** model-routing tooling — a price-performance analyzer, an eval-driven selector, a rule-based router, a semantic difficulty classifier, a cheap-first cascade, a gateway route table with fallbacks, and an A/B-plus-drift monitor — as runnable models.

- **Compare** the routing strategies (rules vs classifier vs cascade vs all-frontier) on cost, overhead, and accuracy, and a cheap model against a frontier model on a per-task eval.

- **Operate** a confidence threshold in a cascade, a fallback chain in a gateway, and a per-route quality monitor.

- **Evaluate** a router: did it cut cost meaningfully AND hold quality on every route, and is any route drifting?

> 📦 **Info**
>
> ✅ Before you startIn **13.1** you learned the same request can cost roughly twenty-four times more on a frontier model than a small one — tier by difficulty. This lesson turns that one insight into a whole discipline. In **13.2** you kept a cheaper prompt only if quality held on an eval set; a router is the same bet at the model level. And in **12.3** the AI gateway was the one place that fronts every provider and retries — that is exactly where a router lives. Latency (13.4) and self-hosting a route target (13.5 / 13.6) come later in this module.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 🚑 **Analogy**
>
> Model routing is **hospital triage**. A hospital does not send every patient to the brain surgeon: a triage nurse sees the headache and sends it to the GP, the fracture to orthopedics, and only the genuine emergency to the specialist. Sending everyone to the surgeon would be ruinously expensive and would not even help the headache. A router is that triage desk for queries: the greeting goes to a small model, the summary to a mid model, and only the hard reasoning to the frontier. **Where it breaks down:** a hospital triage nurse is a trained human, while your router is code — so you have to *measure* whether it is sorting correctly, which is the last step of this lesson.

> 📦 **Info**
>
> 🚫 Two misconceptions this lesson kills
> - **“The best model for everything is the biggest model.”** Most queries are easy; the price spread is twenty to a hundred times for a tiny quality gap. The best model is the *cheapest one that clears the bar* for that query.
> - **“A router is set-and-forget.”** A route drifts when a model changes upstream — its quality falls while the bill still looks great. You must measure per-route quality, not just the total cost.

> 💡 **Info**
>
> ⚠️ Anti-patternTwo habits that quietly undo your work. **Routing without a per-task eval** — you cannot know the cheap model is good enough until you measure it, so a blind route silently ships worse answers. And **running a cascade with too high an escalation rate** — because escalated queries pay for both models, past a break-even point the cascade costs *more* than just using the frontier for everything.

---

## 🎯 Concept 1: Why One Model For Everything Is Wrong

### Why One Model For Everything Is Wrong

The model market spans a 20-100x price range from a nano tier to a frontier tier, but the quality gap on an easy query is tiny. Sending every request to the frontier overpays on the easy majority. Matching the model to task difficulty is the lever - teams routinely cut cost 40-70 percent.

Start with the shape of the market. In 2026 the price of a model spans a **twenty-to-a-hundred-fold range**: a nano or flash-lite tier costs a few cents per million tokens, a frontier tier costs several dollars. The tiers run across the providers — a nano tier like Google’s Gemini Flash-Lite or OpenAI’s GPT nano, a small tier like Anthropic’s Claude Haiku, up to a frontier tier like Anthropic’s Claude Opus or OpenAI’s GPT (illustrative examples; verify live names and prices). But here is the asymmetry that makes routing work: on an *easy* query — a greeting, a classification, a short factual answer — the quality gap between the cheapest model and the frontier is almost nothing. So sending every request to the frontier model marks up the easy majority of your traffic by that huge multiple for a quality difference you could not measure if you tried. The lever is to **match the model to the task’s difficulty**: keep the frontier for the genuinely hard queries and send the rest to a small model. Teams that do this routinely cut inference cost **forty to seventy percent** at the same quality on the easy queries, which is why thirty-seven percent of enterprises now run five or more models in production. The block prices the three strategies, keyless (illustrative tier table).

> ✈️ **Analogy**
>
> It is **flying first class to every destination, even the corner shop**. First class is worth it on a fourteen-hour intercontinental flight; it is absurd for the two-minute hop to the shop on the next street, where a walk gets you there just as well. The frontier model is that first-class seat: worth its price for the genuinely hard query, pure waste for “what are your hours?” Routing is choosing the right way to travel for each trip — walk, bus, or first-class flight — instead of booking first class for your entire life.

A workload runs ten thousand requests a day. Sending all of them to the frontier model costs about a hundred and twenty-five dollars a day. If you send the easy 70 percent to a small model, the medium 20 percent to a mid model, and the hard 10 percent to frontier, roughly what happens to the cost?

**📝 `01_price_performance_spread.py`** - *runnable*

In [ ]:
TIERS = {                       # illustrative $/1M tokens + eval-score; NOT live prices
    "frontier": {"in": 5.00, "out": 25.00, "q": 0.95},   # opus / GPT-5-class
    "mid":      {"in": 3.00, "out": 15.00, "q": 0.90},   # sonnet-class
    "small":    {"in": 1.00, "out":  5.00, "q": 0.82},   # haiku-class
    "nano":     {"in": 0.25, "out":  1.50, "q": 0.72},   # flash-lite / nano-class
}
def cost(tier, tin, tout):      # per-request cost in USD
    return (tin * TIERS[tier]["in"] + tout * TIERS[tier]["out"]) / 1_000_000
# Why "one model for everything" is the wrong default: the price-performance SPREAD.
TIN, TOUT, REQ = 1000, 300, 10000        # one workload: 1000 in / 300 out, 10k requests/day
print("Per-request cost by tier (in {} / out {} tokens):".format(TIN, TOUT))
for t in TIERS:
    print("  {:<9} ${:.4f}/req   quality {:.2f}".format(t, cost(t, TIN, TOUT), TIERS[t]["q"]))
print()
all_frontier = cost("frontier", TIN, TOUT) * REQ
all_small    = cost("small", TIN, TOUT) * REQ
# matched: send the easy 70% to small, the medium 20% to mid, the hard 10% to frontier
matched = (0.70*cost("small",TIN,TOUT) + 0.20*cost("mid",TIN,TOUT) + 0.10*cost("frontier",TIN,TOUT)) * REQ
print("Spread: frontier input is {:.0f}x the nano input price - but the quality gap is only {:.2f}.".format(
    TIERS["frontier"]["in"]/TIERS["nano"]["in"], TIERS["frontier"]["q"]-TIERS["nano"]["q"]))
print("At {:,} req/day:".format(REQ))
print("  everything on frontier: ${:,.2f}/day   (safe, but you overpay on every easy query)".format(all_frontier))
print("  everything on small:    ${:,.2f}/day   ({:.0f}x cheaper - but quality drops on the hard ones)".format(all_small, all_frontier/all_small))
print("  matched to difficulty:  ${:,.2f}/day   ({:.1f}x cheaper, frontier kept for the hard 10%)".format(matched, all_frontier/matched))
print()
print("Most queries do not need a frontier model. Matching the model to the task is the lever this lesson pulls.")

# Output:
# Per-request cost by tier (in 1000 / out 300 tokens):
#   frontier  $0.0125/req   quality 0.95
#   mid       $0.0075/req   quality 0.90
#   small     $0.0025/req   quality 0.82
#   nano      $0.0007/req   quality 0.72
#
# Spread: frontier input is 20x the nano input price - but the quality gap is only 0.23.
# At 10,000 req/day:
#   everything on frontier: $125.00/day   (safe, but you overpay on every easy query)
#   everything on small:    $25.00/day   (5x cheaper - but quality drops on the hard ones)
#   matched to difficulty:  $45.00/day   (2.8x cheaper, frontier kept for the hard 10%)
#
# Most queries do not need a frontier model. Matching the model to the task is the lever this lesson pulls.

- Each tier is priced per request — the **frontier is about five times the small model** and twenty times the nano tier, for a quality gap of only a couple of points.
- **Everything on frontier** is the safe, expensive default; **everything on small** is far cheaper but drops quality on the hard queries.
- **Matched to difficulty** — easy to small, medium to mid, hard to frontier — is several times cheaper than all-frontier while keeping the frontier for the hard slice.
- The lesson: most queries do not need a frontier model, so matching the model to the task is the biggest cost lever left after trimming tokens.

#### 💡 What just happened

⚡ What just happened? The model market spans a 20-100x price range, but the quality gap on an easy query is tiny - so sending everything to the frontier overpays the easy majority. Matching the model to difficulty cuts cost 40-70 percent at the same quality. Tradeoff / the framing: you take on the complexity of a router in exchange for that saving. Next: how you decide which cheaper model is actually good enough.

- A difficulty-mix slider moves traffic across the frontier / mid / small / nano tiers
- The daily-cost bar falls from all-frontier toward matched while a quality gauge barely moves

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Select The Model: Measure, Do Not Guess

### Select The Model: Measure, Do Not Guess

Selection is a measurement problem. Build a small eval set per task, score each candidate model on it, and pick the cheapest model that clears your quality bar. The frontier model’s extra couple of points are often not worth several times the price for a given task.

Before you can route, you have to know which model is good enough for what — and that is a **measurement**, not a guess. The method: for a given task, build a small **eval set** (a couple of dozen representative examples with known-good answers), run each candidate model on it, score the quality, and note the cost. Now you have a **quality-versus-cost frontier** for that task, and the rule is simple: **pick the cheapest model that clears your quality bar**. Often a small model scores just above the bar at a fraction of the frontier’s price, and the frontier’s extra couple of points of quality are not worth several times the cost *for that task*. This is exactly what RouterBench does at scale — hundreds of thousands of precomputed outcomes across eleven models and seven tasks — but you do it in miniature per task. Selection is per-task because a model that is excellent at summarizing can be mediocre at code. The block runs the eval and picks, keyless.

> 🤝 **Analogy**
>
> It is **hiring the cheapest candidate who passes the test**, not the one with the most impressive resume. You do not pay a principal engineer’s salary to answer the support line if a junior aces the same screening test — the extra credentials are real, but wasted on this job. Selecting a model is that hiring decision: you give every candidate the same task-specific test (your eval set), and you hire the cheapest one that passes, because the frontier model’s extra polish is not worth its salary on a task a small model already handles.

On a task, you measure: nano 0.71, small 0.86, mid 0.91, frontier 0.94. Your quality bar is 0.85. Which model do you pick?

**📝 `02_eval_driven_selection.py`** - *runnable*

In [ ]:
TIERS = {                       # illustrative $/1M tokens + eval-score; NOT live prices
    "frontier": {"in": 5.00, "out": 25.00, "q": 0.95},   # opus / GPT-5-class
    "mid":      {"in": 3.00, "out": 15.00, "q": 0.90},   # sonnet-class
    "small":    {"in": 1.00, "out":  5.00, "q": 0.82},   # haiku-class
    "nano":     {"in": 0.25, "out":  1.50, "q": 0.72},   # flash-lite / nano-class
}
def cost(tier, tin, tout):      # per-request cost in USD
    return (tin * TIERS[tier]["in"] + tout * TIERS[tier]["out"]) / 1_000_000
# Selection = measure, do not guess. Build a small eval set per TASK, score each model, pick the
# CHEAPEST model that clears your quality bar (the RouterBench idea).
QUALITY_BAR = 0.85
TIN, TOUT = 1000, 300
# measured eval scores for THIS task (illustrative; from running a 20-example eval set per model)
measured = {"nano": 0.71, "small": 0.86, "mid": 0.91, "frontier": 0.94}
print("Task eval (quality bar = {:.2f}) - cheapest model that CLEARS the bar wins:".format(QUALITY_BAR))
pick = None
for t in ["nano", "small", "mid", "frontier"]:   # cheapest -> most expensive
    q, c = measured[t], cost(t, TIN, TOUT)
    ok = q >= QUALITY_BAR
    if ok and pick is None: pick = t
    print("  {:<9} quality {:.2f}  ${:.4f}/req  -> {}".format(t, q, c, "CLEARS the bar" if ok else "below the bar"))
print()
front_c, pick_c = cost("frontier", TIN, TOUT), cost(pick, TIN, TOUT)
print("Pick: {} at quality {:.2f}, ${:.4f}/req.".format(pick, measured[pick], pick_c))
print("  vs defaulting to frontier ({:.2f}, ${:.4f}): {:.0f}x cheaper, and the bar is still met.".format(
    measured["frontier"], front_c, front_c/pick_c))
print("  frontier's extra {:.2f} quality is not worth {:.0f}x the price for THIS task.".format(
    measured["frontier"]-measured[pick], front_c/pick_c))
print("Selection is per-task: run the eval, plot quality vs cost, keep the cheapest that clears the bar.")

# Output:
# Task eval (quality bar = 0.85) - cheapest model that CLEARS the bar wins:
#   nano      quality 0.71  $0.0007/req  -> below the bar
#   small     quality 0.86  $0.0025/req  -> CLEARS the bar
#   mid       quality 0.91  $0.0075/req  -> CLEARS the bar
#   frontier  quality 0.94  $0.0125/req  -> CLEARS the bar
#
# Pick: small at quality 0.86, $0.0025/req.
#   vs defaulting to frontier (0.94, $0.0125): 5x cheaper, and the bar is still met.
#   frontier's extra 0.08 quality is not worth 5x the price for THIS task.
# Selection is per-task: run the eval, plot quality vs cost, keep the cheapest that clears the bar.

- Each candidate is scored on the **task’s eval set** and shown with its cost — the nano tier is below the bar, so it is out.
- The **cheapest model that clears the quality bar** wins — here the small model at 0.86, several times cheaper than frontier.
- Frontier’s extra couple of points of quality is real, but **not worth several times the price** for this task.
- The lesson: you do not guess which model is good enough — you measure a per-task eval and keep the cheapest that clears the bar.

#### 💡 What just happened

⚡ What just happened? Selection is measurement: build a per-task eval, score each candidate, and pick the cheapest model that clears your quality bar - the frontier’s extra polish is often not worth several times the price. Tradeoff: building the eval set costs you some up-front work, in exchange for a defensible, per-task model choice. Next: the cheapest way to actually route a live query - rules.

- A quality-vs-cost scatter of the four tiers with a draggable quality-bar line
- The cheapest model above the line lights up as the pick; defaulting to frontier shows the wasted multiple

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: Rule-Based Routing

### Rule-Based Routing

The cheapest router is a lookup table of rules on cheap signals - task-type keywords, input length - evaluated before the request at about zero milliseconds. Rules catch the obvious cases for free, but they are brittle: a short query can still be hard, so rules are the floor, not the whole story.

Now route a live query, starting with the cheapest possible router: **rules**. A rule-based router is a small lookup table evaluated **before** you call any model — it reads cheap signals off the query (its **task-type keywords**, its **input length**, structured fields) and picks a tier. “Contains ‘classify’ or ‘extract’ and is short” goes to the small model; “longer than two thousand tokens” goes to the frontier for the context; a reasoning keyword goes to mid. Because it is just string and length checks, it runs at essentially **zero milliseconds** and zero cost, and it correctly handles a large fraction of everyday traffic. The catch is that rules are **brittle**: a short query can still be hard (a one-line proof), and a long one can be trivial (a padded greeting), so rules alone will misroute the tricky cases. That is why rules are the **floor** — the free first pass — not the whole router. The block routes a batch by rules, keyless.

> 📋 **Analogy**
>
> It is **a triage nurse with a clipboard**. The nurse does not run tests on everyone; a checklist sorts the obvious arrivals in seconds — visible bleeding to the front, a cough to the waiting room — for free and instantly. That clipboard is the rule router: a fixed set of “if this, then that” checks that handle the clear-cut cases at a glance. And just like the nurse, the clipboard cannot catch everything: the quiet patient with a subtle but serious problem needs a closer look, which is the next layer.

A rule router sends any query containing ‘classify’ or ‘extract’ to the small model. A user sends: “Classify whether this 40-page merger contract triggers antitrust review.” What happens?

**📝 `03_rule_based_routing.py`** - *runnable*

In [ ]:
TIERS = {                       # illustrative $/1M tokens + eval-score; NOT live prices
    "frontier": {"in": 5.00, "out": 25.00, "q": 0.95},   # opus / GPT-5-class
    "mid":      {"in": 3.00, "out": 15.00, "q": 0.90},   # sonnet-class
    "small":    {"in": 1.00, "out":  5.00, "q": 0.82},   # haiku-class
    "nano":     {"in": 0.25, "out":  1.50, "q": 0.72},   # flash-lite / nano-class
}
def cost(tier, tin, tout):      # per-request cost in USD
    return (tin * TIERS[tier]["in"] + tout * TIERS[tier]["out"]) / 1_000_000
# Rule-based routing: the CHEAPEST router (~0 ms). Route on simple query signals - task type, length -
# with a lookup table. Pre-request rules catch the obvious cases for free.
TOUT = 300
def route_rules(query, tin):
    q = query.lower()
    if tin > 2000:                                   return "frontier"  # very long context -> strongest
    if any(k in q for k in ("prove", "design", "debug", "analyze")):  return "mid"   # reasoning-ish
    if any(k in q for k in ("classify", "extract", "format", "tag")): return "small" # mechanical
    return "small"                                   # default: assume easy until shown otherwise
queries = [
    ("Classify this ticket as bug or feature", 120),
    ("Extract the invoice total and date", 90),
    ("Design a schema for multi-tenant billing", 300),
    ("Summarize this 4000-token contract", 4000),
    ("Tag the sentiment of this review", 60),
]
print("Rule-based routing (0 ms overhead - just string + length checks):")
routed = 0.0
for text, tin in queries:
    t = route_rules(text, tin)
    c = cost(t, tin, TOUT)
    routed += c
    print("  {:<9} <- {}".format(t, text[:38]))
front = sum(cost("frontier", tin, TOUT) for _, tin in queries)
print()
print("Batch cost: routed ${:.4f} vs all-frontier ${:.4f}  = {:.1f}x cheaper, decided before the call.".format(
    routed, front, front/routed))
print("Rules are brittle (a short query can still be hard), so they are the FLOOR, not the whole story.")

# Output:
# Rule-based routing (0 ms overhead - just string + length checks):
#   small     <- Classify this ticket as bug or feature
#   small     <- Extract the invoice total and date
#   mid       <- Design a schema for multi-tenant billi
#   frontier  <- Summarize this 4000-token contract
#   small     <- Tag the sentiment of this review
#
# Batch cost: routed $0.0377 vs all-frontier $0.0604  = 1.6x cheaper, decided before the call.
# Rules are brittle (a short query can still be hard), so they are the FLOOR, not the whole story.

- Each query is routed by **keyword and length rules** alone — no model call, so the overhead is essentially zero.
- The batch is **cheaper than all-frontier** because the mechanical queries (classify, extract, tag) drop to the small model.
- The rules are **decided before the call**, which is what makes them free — they are the cheapest layer you can add.
- But the rules are **brittle** — a short-but-hard query slips through — so they are the floor, and the next layer catches what they miss.

#### 💡 What just happened

⚡ What just happened? Rule-based routing reads cheap signals (keywords, length) before the call at ~0ms and free, catching the obvious cases - but it is brittle, so a short-but-hard query slips through. Tradeoff: near-zero cost and latency, in exchange for coverage gaps on the tricky queries. Next: a classifier that reads the query’s content to catch what the rules miss.

- Queries drop onto a clipboard of rules (keyword, length) and sort into tier lanes at 0ms
- A short-but-hard query visibly lands in the wrong lane - rules are brittle

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Semantic / Classifier Routing

### Semantic / Classifier Routing

When rules are too brittle, route on meaning. A semantic router embeds the query and routes by similarity, or a small classifier scores its difficulty. It reads content, so it catches the short-but-hard query. Overhead is small: embedding about 5ms, a classifier 50-100ms, against a 500-2000ms model response.

When the rules misroute a case, you route on the query’s **meaning** instead of its surface. Two flavors. A **semantic router** encodes the query and each candidate route into embeddings and sends the query to the closest one — intent matching, not keyword matching. A **difficulty classifier** is a small model (or a learned scorer) that reads the query and outputs a difficulty score, which you map to a tier. Either way, it reads the *content*, so it catches the case rules cannot: a **short query that is actually hard** (a twelve-word request to prove a concurrency invariant) gets lifted to the frontier even though a length rule would have sent it to the small model. The cost is a little latency, and it is tiny in context: **embedding-based routing adds about five milliseconds and a heavier classifier fifty to a hundred**, against a model response that takes five hundred to two thousand. So you pay a rounding error in latency to stop misrouting the hard queries. The block scores difficulty and routes, keyless.

> 🎟️ **Analogy**
>
> It is **a receptionist who actually listens to what you are asking**, instead of routing you by the job title on your badge. A keyword rule is the badge reader: “engineering badge, third floor” — fast, but it sends the engineer with a payroll question to the wrong place. A good receptionist hears “I have a question about my paycheck” and routes on the *content*, straight to HR. The difficulty classifier is that receptionist: it listens to what the query is really asking and routes accordingly, catching the ones the badge reader gets wrong.

A length rule would send the 12-word query “Prove this lock is safe under a network partition” to the small model. A difficulty classifier scores its content at 1.0 (very hard). Where does the classifier route it?

**📝 `04_semantic_routing.py`** - *runnable*

In [ ]:
TIERS = {                       # illustrative $/1M tokens + eval-score; NOT live prices
    "frontier": {"in": 5.00, "out": 25.00, "q": 0.95},   # opus / GPT-5-class
    "mid":      {"in": 3.00, "out": 15.00, "q": 0.90},   # sonnet-class
    "small":    {"in": 1.00, "out":  5.00, "q": 0.82},   # haiku-class
    "nano":     {"in": 0.25, "out":  1.50, "q": 0.72},   # flash-lite / nano-class
}
def cost(tier, tin, tout):      # per-request cost in USD
    return (tin * TIERS[tier]["in"] + tout * TIERS[tier]["out"]) / 1_000_000
# Semantic / classifier routing (~5-100 ms): score the query's DIFFICULTY from features a small model
# reads, then route by threshold. Catches hard queries that the length/keyword rules miss.
TIN, TOUT = 800, 300
def difficulty(query):                 # a tiny bag-of-signals classifier (stand-in for an embedding model)
    q = query.lower(); score = 0.30
    for kw, w in [("prove",0.4),("why",0.25),("trade-off",0.3),("edge case",0.35),
                  ("step by step",0.3),("ambiguous",0.3),("nuance",0.3)]:
        if kw in q: score += w
    return min(score, 1.0)
def route_semantic(query):
    d = difficulty(query)
    return ("frontier" if d >= 0.7 else "mid" if d >= 0.5 else "small"), d
cases = [
    "Prove why this distributed lock is correct under a network partition edge case",  # SHORT but HARD
    "What is the capital of France",
    "Explain the nuance in this ambiguous requirement step by step",
]
print("Semantic routing (~50 ms classifier vs 500-2000 ms model response = negligible):")
for text in cases:
    t, d = route_semantic(text)
    print("  difficulty {:.2f} -> {:<9} <- {}".format(d, t, text[:42]))
print()
print("A rule on LENGTH alone sends the 12-word proof query to 'small' and gets a wrong answer;")
print("the difficulty score reads the CONTENT and lifts it to frontier. Classifiers generalize past brittle rules.")

# Output:
# Semantic routing (~50 ms classifier vs 500-2000 ms model response = negligible):
#   difficulty 1.00 -> frontier  <- Prove why this distributed lock is correct
#   difficulty 0.30 -> small     <- What is the capital of France
#   difficulty 1.00 -> frontier  <- Explain the nuance in this ambiguous requi
#
# A rule on LENGTH alone sends the 12-word proof query to 'small' and gets a wrong answer;
# the difficulty score reads the CONTENT and lifts it to frontier. Classifiers generalize past brittle rules.

- A small **difficulty classifier** scores each query’s content and maps the score to a tier — it reads meaning, not surface.
- The **short-but-hard query** (a twelve-word proof) scores high and is lifted to the frontier, where a length rule would have sent it to the small model and gotten a wrong answer.
- The **latency cost is tiny** — about fifty milliseconds for the classifier against a five-hundred-to-two-thousand-millisecond model response.
- The lesson: a classifier generalizes past brittle rules by routing on what the query *means*, catching the cases rules miss.

#### 💡 What just happened

⚡ What just happened? Semantic / classifier routing routes on the query’s content, so it catches the short-but-hard query a length rule misroutes - for a tiny latency cost (~5-100ms vs a 500-2000ms response). Tradeoff: a little latency and a classifier to maintain, in exchange for coverage rules cannot give. Next: escalating only the hard queries with a cheap-first cascade.

- The same short-but-hard query fed to a difficulty meter that reads its content
- The needle climbs and re-routes it up to frontier; a latency readout shows ~50ms vs a 500-2000ms response

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Cascade Routing

### Cascade Routing

A cascade runs the cheap model first, reads a confidence signal, and escalates to a strong model only when confidence is low. It is the most accurate per dollar - but escalated queries pay twice, so if the escalation rate is too high the cascade costs more than all-frontier. The confidence threshold is the knob.

The routers so far decide *before* the call. A **cascade** decides *during*: it runs the **cheap model first**, reads a **confidence signal** off that answer, and **escalates** to a stronger model only when confidence is below a threshold. This is the FrugalGPT pattern, and it is the most accurate per dollar because the easy queries — most of them — stop at the cheap model, while only the genuinely uncertain ones pay for the frontier. The catch is a real one: an escalated query **pays twice**, once for the cheap model and again for the frontier. So the **escalation rate** decides everything. At a modest rate the cascade is far cheaper than all-frontier; but past a break-even point, the double-pay makes the cascade cost *more* than just sending everything to the frontier in the first place. The **confidence threshold is the knob**: raise it and you escalate more (safer, pricier); lower it and you escalate less (cheaper, riskier). The block runs the cascade and shows the double-pay, keyless.

> 🧑‍⚕️ **Analogy**
>
> It is **asking the junior first and escalating to the specialist only when they are unsure**. Most tickets, the junior handles — cheap and fast. When the junior says “I am not confident on this one,” it goes up to the specialist. That escalation is efficient *as long as it is rare*: if the junior punts half of everything upstairs, you are paying the junior’s time *and* the specialist’s on the same tickets, and you would have been cheaper sending them straight to the specialist. The confidence threshold is how sure the junior has to be before they are allowed to answer.

A cascade runs the small model first and escalates on low confidence. Over 100 queries it escalates 25. Each escalated query pays the small model AND the frontier. How does the cost compare to all-frontier?

**📝 `05_cascade_routing.py`** - *runnable*

In [ ]:
TIERS = {                       # illustrative $/1M tokens + eval-score; NOT live prices
    "frontier": {"in": 5.00, "out": 25.00, "q": 0.95},   # opus / GPT-5-class
    "mid":      {"in": 3.00, "out": 15.00, "q": 0.90},   # sonnet-class
    "small":    {"in": 1.00, "out":  5.00, "q": 0.82},   # haiku-class
    "nano":     {"in": 0.25, "out":  1.50, "q": 0.72},   # flash-lite / nano-class
}
def cost(tier, tin, tout):      # per-request cost in USD
    return (tin * TIERS[tier]["in"] + tout * TIERS[tier]["out"]) / 1_000_000
# Cascade routing (FrugalGPT): run the CHEAP model first, read a confidence signal, and ESCALATE to a
# strong model ONLY when confidence is low. Most accurate per dollar - but escalated queries pay TWICE.
CONF_THRESHOLD = 0.70
TIN, TOUT, N = 900, 300, 100
# illustrative per-query confidence from the small model; the ~25% below the threshold escalate
low  = [0.52, 0.58, 0.63, 0.67, 0.69] * 5     # 25 low-confidence  -> escalate to frontier
high = [0.74, 0.80, 0.86, 0.91, 0.96] * 15    # 75 high-confidence -> stop at small
confidences = (low + high)[:N]
small_c, front_c = cost("small", TIN, TOUT), cost("frontier", TIN, TOUT)
escalated = sum(1 for c in confidences if c < CONF_THRESHOLD)
# every query pays the small model; the low-confidence ones ALSO pay the frontier model
cascade_cost = N * small_c + escalated * front_c
all_front = N * front_c
print("Cascade over {} queries, escalate when confidence < {:.2f}:".format(N, CONF_THRESHOLD))
print("  escalated: {}/{} queries ({:.0f}%) went small -> frontier".format(escalated, N, escalated/N*100))
print("  cost: cascade ${:.4f}  vs all-frontier ${:.4f}  = {:.1f}x cheaper".format(cascade_cost, all_front, all_front/cascade_cost))
print("  the {} hard queries STILL got the frontier answer - only the easy {} stopped at small.".format(escalated, N-escalated))
print()
print("Watch the double-pay: an escalated query costs small + frontier (${:.4f}). If you escalate too often,".format(small_c+front_c))
print("cascade costs MORE than all-frontier. The confidence THRESHOLD is the knob: higher = safer + pricier.")

# Output:
# Cascade over 100 queries, escalate when confidence < 0.70:
#   escalated: 25/100 queries (25%) went small -> frontier
#   cost: cascade $0.5400  vs all-frontier $1.2000  = 2.2x cheaper
#   the 25 hard queries STILL got the frontier answer - only the easy 75 stopped at small.
#
# Watch the double-pay: an escalated query costs small + frontier ($0.0144). If you escalate too often,
# cascade costs MORE than all-frontier. The confidence THRESHOLD is the knob: higher = safer + pricier.

- Every query runs the **small model first**; the ones whose confidence is below the threshold **escalate** to the frontier.
- At the shown escalation rate the cascade is **about twice as cheap** as all-frontier, and the hard queries still get the frontier answer.
- The **double-pay** is the risk: an escalated query costs the small model plus the frontier, so heavy escalation flips the saving.
- The **confidence threshold is the knob** — higher means more escalation (safer, pricier), lower means less (cheaper, riskier).

#### 💡 What just happened

⚡ What just happened? A cascade runs the cheap model first and escalates only low-confidence queries, so the easy majority stops cheap - but escalated queries pay twice, so too high an escalation rate costs more than all-frontier. Tradeoff: the confidence threshold trades cost against safety. Next: where the router actually lives in production - the gateway.

- A confidence-threshold slider; queries run the cheap model first and low-confidence ones escalate
- The cost bar shows the double-pay - drag the threshold too high and cascade overtakes all-frontier

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: The Routing Gateway + Fallback

### The Routing Gateway + Fallback

In production the router lives in the AI gateway (12.3): one place holds the route table and a fallback chain per model, so a provider error, rate-limit, or context overflow degrades to a slower answer on the next model rather than a failed request. LiteLLM ships three fallback types.

All of this has to live *somewhere* in production, and that somewhere is the **AI gateway** from Lesson 12.3. The gateway holds the **route table** (task or tier to model) in one place, and — just as importantly — a **fallback chain** per model. When the primary model for a route fails (a provider error, a **rate-limit**, a **context-window overflow**, a content-policy block), the gateway does not return an error to the user: it **hops to the next model** in that route’s chain, degrading to a slightly slower or stronger answer rather than a dropped request. LiteLLM makes this concrete with three fallback types — general, `context_window_fallbacks`, and `content_policy_fallbacks` — and OpenRouter, Portkey, Martian, and Not Diamond offer managed versions. Putting routing in the gateway also means one place to **meter cost per route**, which you will need for the last step. The block runs a route table with fallbacks, keyless.

> ☎️ **Analogy**
>
> It is **a switchboard with backup lines**. When you call the main number and it is busy, a good switchboard does not just give you a dead tone — it rolls the call to the next available line, so you still get through, maybe to a different desk. The gateway is that switchboard: the route table says which ‘line’ (model) to try first, and the fallback chain is the backup lines it rolls to when the first is busy or down. The caller gets an answer, not a dropped call.

Your gateway routes ‘summarize’ to the mid model, with a fallback chain [frontier]. The mid model’s provider starts returning rate-limit errors. What does the user get?

**📝 `06_gateway_fallback.py`** - *runnable*

In [ ]:
TIERS = {                       # illustrative $/1M tokens + eval-score; NOT live prices
    "frontier": {"in": 5.00, "out": 25.00, "q": 0.95},   # opus / GPT-5-class
    "mid":      {"in": 3.00, "out": 15.00, "q": 0.90},   # sonnet-class
    "small":    {"in": 1.00, "out":  5.00, "q": 0.82},   # haiku-class
    "nano":     {"in": 0.25, "out":  1.50, "q": 0.72},   # flash-lite / nano-class
}
def cost(tier, tin, tout):      # per-request cost in USD
    return (tin * TIERS[tier]["in"] + tout * TIERS[tier]["out"]) / 1_000_000
# The routing gateway (Lesson 12.3) is where routing LIVES in production: a route table plus a fallback
# chain per model, so a provider error / rate-limit / context-overflow degrades to a slower answer, not a 500.
route_table = {"classify": "small", "summarize": "mid", "reason": "frontier"}
fallbacks = {"small": ["mid", "frontier"], "mid": ["frontier"], "frontier": ["mid"]}   # try in order on failure
def serve(task, failing):                 # failing = set of tiers currently erroring / rate-limited
    chain = [route_table[task]] + fallbacks[route_table[task]]
    for tier in chain:
        if tier not in failing:
            return tier, chain.index(tier)
    return None, len(chain)
requests = [("classify", set()), ("summarize", {"mid"}), ("reason", {"frontier"}), ("classify", {"small","mid"})]
print("Gateway route table + fallback chain (degrade, do not drop):")
for task, failing in requests:
    tier, hops = serve(task, failing)
    note = "primary" if hops == 0 else "fallback after {} hop(s)".format(hops)
    fail_note = " (failing: {})".format(sorted(failing)) if failing else ""
    print("  {:<10} -> {:<9} [{}]{}".format(task, tier, note, fail_note))
print()
print("LiteLLM ships 3 fallback types: general, context_window, and content_policy.")
print("Routing is a GATEWAY concern: one place decides the model, retries on failure, and meters cost per route.")

# Output:
# Gateway route table + fallback chain (degrade, do not drop):
#   classify   -> small     [primary]
#   summarize  -> frontier  [fallback after 1 hop(s)] (failing: ['mid'])
#   reason     -> mid       [fallback after 1 hop(s)] (failing: ['frontier'])
#   classify   -> frontier  [fallback after 2 hop(s)] (failing: ['mid', 'small'])
#
# LiteLLM ships 3 fallback types: general, context_window, and content_policy.
# Routing is a GATEWAY concern: one place decides the model, retries on failure, and meters cost per route.

- A **route table** maps each task to its primary model, and each model has a **fallback chain** to try in order on failure.
- When a model is **failing** (error / rate-limit / context overflow), the request **hops to the next model** in the chain instead of dropping.
- A request can take **more than one hop** if several models are down, ending on whichever is healthy — degrade, do not drop.
- The gateway is the one place that decides the model, retries on failure (LiteLLM has three fallback types), and meters cost per route.

#### 💡 What just happened

⚡ What just happened? In production the router lives in the gateway (12.3): a route table plus a per-model fallback chain, so a provider failure degrades to the next model instead of dropping the request. Tradeoff: a fallback answer may be slower or pricier, in exchange for not failing the user. And the gateway is where you meter cost per route - which the last step needs.

- A switchboard route table; toggle a provider to failing
- The request hops down its fallback chain to the next tier instead of dropping, served model lit

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: Measure The Router

### Measure The Router

A router is a bet, so you A/B it against the all-frontier baseline on cost AND quality and keep it only if quality holds. The subtle failure is drift: a route’s model degrades upstream, its quality falls, and the bill still looks great. Track per-route quality, not just the total, and alert on a breach.

Every routing decision so far *risks quality*, so the final discipline is to never trust a router blind. You **A/B it against the all-frontier baseline** — the safe, expensive default — on **both cost and quality**, and you keep the router only if its quality holds within tolerance. A router that is sixty-four percent cheaper but drops below your quality floor is a *net loss*, not a win. But the harder failure is **drift**: a route’s model gets quietly deprecated or degraded upstream, that route’s quality falls, and because the *bill still looks great*, nobody notices the answers getting worse. So you track **per-route quality**, not just the aggregate, and you alert when any single route breaches its floor. This is the same “measure the tradeoff” discipline from 13.2 and the online-eval loop from 12.8, applied at the model-routing level. Even a learned router like RouteLLM — which reports about ninety-five percent of frontier quality at roughly eighty-five percent lower cost — has to be monitored, because the traffic and the models both move. The block runs the A/B and the drift watch, keyless.

> 🌡️ **Analogy**
>
> It is **a thermostat you actually check, not one you set and forget**. You set it to a comfortable temperature and it works — until a window cracks open, the reading drifts, and the room is cold while the thermostat still *reports* the number you set. If you never look, you never notice. Measuring a router is checking the actual room temperature: the router’s dashboard says ‘sixty-four percent cheaper’ and looks great, but you also watch each route’s *quality*, because that is the reading that silently drifts when a model changes upstream.

Your router is 64 percent cheaper than all-frontier and its weighted quality is 0.896, above your 0.88 floor. A month later the small route’s model is quietly changed upstream and its quality drops, pulling the weighted quality to 0.840. The cost is still 64 percent lower. What should happen?

**📝 `07_measure_the_router.py`** - *runnable*

In [ ]:
TIERS = {                       # illustrative $/1M tokens + eval-score; NOT live prices
    "frontier": {"in": 5.00, "out": 25.00, "q": 0.95},   # opus / GPT-5-class
    "mid":      {"in": 3.00, "out": 15.00, "q": 0.90},   # sonnet-class
    "small":    {"in": 1.00, "out":  5.00, "q": 0.82},   # haiku-class
    "nano":     {"in": 0.25, "out":  1.50, "q": 0.72},   # flash-lite / nano-class
}
def cost(tier, tin, tout):      # per-request cost in USD
    return (tin * TIERS[tier]["in"] + tout * TIERS[tier]["out"]) / 1_000_000
# Never route blind: a router is a BET. A/B it against the all-frontier baseline on cost AND quality,
# keep it only if quality holds, and watch EACH route for silent drift (a model that quietly got worse).
QUALITY_FLOOR = 0.88
TIN, TOUT, REQ = 1000, 300, 10000
mix = {"small": 0.70, "mid": 0.20, "frontier": 0.10}   # traffic share per route
def weighted(quality_by_tier):
    return sum(mix[t] * quality_by_tier[t] for t in mix)
router_cost = sum(mix[t] * cost(t, TIN, TOUT) for t in mix) * REQ
base_cost   = cost("frontier", TIN, TOUT) * REQ
healthy = {"small": 0.88, "mid": 0.92, "frontier": 0.96}   # per-route quality, measured on the traffic each gets
rq = weighted(healthy)
verdict = "KEEP the router" if rq >= QUALITY_FLOOR else "REJECT (quality below floor)"
print("A/B the router vs all-frontier (quality floor = {:.2f}):".format(QUALITY_FLOOR))
print("  router:       ${:,.2f}/day  quality {:.3f}".format(router_cost, rq))
print("  all-frontier: ${:,.2f}/day  quality {:.3f}".format(base_cost, healthy["frontier"]))
print("  -> {:.0f}% cheaper, quality {:.3f} vs {:.2f} floor -> {}.".format((1-router_cost/base_cost)*100, rq, QUALITY_FLOOR, verdict))
print()
# Silent drift: the 'small' route's model is deprecated / quietly degrades from 0.88 -> 0.80
drifted = dict(healthy); drifted["small"] = 0.80
dq = weighted(drifted)
alert = "floor BREACHED -> alert + reroute" if dq < QUALITY_FLOOR else "still above floor"
print("Drift watch: the 'small' route quietly drops 0.88 -> 0.80 (a model change upstream).")
print("  weighted router quality falls {:.3f} -> {:.3f}  vs {:.2f} floor -> {}.".format(rq, dq, QUALITY_FLOOR, alert))
print("Track cost saved AND per-route quality. A bad route degrades silently while the bill still looks great.")

# Output:
# A/B the router vs all-frontier (quality floor = 0.88):
#   router:       $45.00/day  quality 0.896
#   all-frontier: $125.00/day  quality 0.960
#   -> 64% cheaper, quality 0.896 vs 0.88 floor -> KEEP the router.
#
# Drift watch: the 'small' route quietly drops 0.88 -> 0.80 (a model change upstream).
#   weighted router quality falls 0.896 -> 0.840  vs 0.88 floor -> floor BREACHED -> alert + reroute.
# Track cost saved AND per-route quality. A bad route degrades silently while the bill still looks great.

**📝 `route-pipeline.py`** - *illustrative*

In [ ]:
# PRODUCTION ROUTER - an illustrative minimal example (rules + classifier, behind a gateway with fallbacks).
from litellm import Router          # the gateway that fronts every provider (Lesson 12.3)

router = Router(
    model_list=[                    # illustrative 2026 model IDs; verify live names + prices
        {"model_name": "small",    "litellm_params": {"model": "anthropic/claude-haiku-4-5"}},
        {"model_name": "mid",      "litellm_params": {"model": "anthropic/claude-sonnet-4-6"}},
        {"model_name": "frontier", "litellm_params": {"model": "anthropic/claude-opus-4-8"}},
    ],
    fallbacks=[{"small": ["mid", "frontier"]}, {"mid": ["frontier"]}],   # degrade, do not drop
)

def pick_tier(query, in_tokens):    # rules first (free), then a difficulty classifier
    if in_tokens > 2000 or difficulty(query) >= 0.7:  return "frontier"
    if difficulty(query) >= 0.5:                       return "mid"
    return "small"                                     # default to the cheapest until shown otherwise

def answer(query, in_tokens):
    tier = pick_tier(query, in_tokens)
    resp = router.completion(model=tier, messages=[{"role": "user", "content": query}])
    log_route(tier, resp.usage)     # meter per-route cost AND quality, so you can watch drift (Step 7)
    return resp.choices[0].message.content
# Then A/B this against all-frontier on your eval set; ship it only if EVERY route holds quality.
# Output: (illustrative minimal example - needs litellm + provider keys + an eval set; not run here.)

- The router is **A/B-tested** against the all-frontier baseline on cost AND quality: it is much cheaper and, while every route holds its floor, it is **kept**.
- Then one route **silently drifts** — the small route’s quality drops upstream — and the weighted quality **breaches the floor**, so an alert fires even though the bill still looks great.
- The lesson: track **per-route quality**, not just the total cost, because a bad route degrades silently.
- The **illustrative router** ties it together: rules plus a classifier pick the tier, a gateway with fallbacks serves it, and every call is metered per route so you can watch for drift.

#### 💡 What just happened

⚡ What just happened? A router is a bet: A/B it against all-frontier on cost AND quality, keep it only if every route holds its floor, and watch for a route silently drifting worse while the bill still looks great. Tradeoff: monitoring costs an eval run per route, in exchange for never trading quality for a lower bill. That is the whole lesson: cheaper per query, quality held on every route, measured.

- An A/B panel: the router cost bar drops while a per-route quality gauge holds above a floor (keep)
- Then one route silently drifts down and the weighted quality breaches the floor (alert)

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture
> A router is a pipeline of decisions, each one measured. It starts from the market fact that **the price spread is twenty to a hundred times for a tiny quality gap**, so matching the model to the task is the biggest cost lever left (Step 1). You **select** per task by measuring an eval set and keeping the cheapest model over the bar (Step 2). Then you route a live query in layers: **rules** for the obvious cases at zero cost (Step 3), a **classifier** for the ambiguous ones that reads content not surface (Step 4), and a **cascade** for the quality-critical ones that escalates only on low confidence (Step 5). The whole thing lives in the **gateway**, with a fallback chain so a provider outage degrades instead of drops (Step 6). And you **measure the router itself**, holding every route’s quality floor and watching for drift (Step 7). Ask two questions of any router: did it **cut cost meaningfully**, and did **quality hold on every route**?

| Strategy | Routing overhead | When to reach for it |
|---|---|---|
| Rule-based | ~0 ms, free | obvious, high-volume cases (greeting, classify, extract) |
| Semantic / classifier | ~5-100 ms | ambiguous queries where content beats surface signals |
| Cascade (cheap-first) | a cheap call + a confidence check | quality-critical work, when escalation stays low |
| All-frontier (baseline) | none | the safe, expensive default you A/B every router against |

> 📦 **Info**
>
> ➡️ Where this goes nextYou can now send each query to the cheapest model that answers it well. Routing changes **latency** too — a small model is faster as well as cheaper — which is Lesson 13.4. The **gateway** that holds your route table and fallbacks is Lesson 12.3, and a **self-hosted open model** makes an even cheaper route target once you cross the self-host threshold in Lessons 13.5 and 13.6. Routing also shows up as an LLMOps serving discipline in Lesson 14.5. The primary references are [RouteLLM](https://github.com/lm-sys/RouteLLM) and its [paper](https://arxiv.org/abs/2406.18665), [RouterBench](https://arxiv.org/abs/2403.12031), [FrugalGPT](https://arxiv.org/abs/2305.05176), [LiteLLM routing](https://docs.litellm.ai/docs/routing), and the [awesome-ai-model-routing](https://github.com/Not-Diamond/awesome-ai-model-routing) list.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Model Selection& Routing**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-13_3.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-13.3.html` - regenerate this notebook whenever the source HTML is updated.*
